# D1-06 Data quality checks and linking workflow

This notebook is used after BAFU import (from `D1-04`).


## Learning goals

- Inspect imported Brightway databases.
- Detect simple linking inconsistencies in tabular foreground data.
- Prepare clean foreground data before formal Excel import.


In [ ]:
import bw2data as bd
import pandas as pd
from pathlib import Path

bd.projects.set_current('barcelona-rlcia-2026')

print('Current project:', bd.projects.current)
print('Databases:', list(bd.databases))

if 'bafu-2025' in bd.databases:
    print('Number of activities in bafu-2025:', len(bd.Database('bafu-2025')))
else:
    print("Database 'bafu-2025' not found. Complete D1-04 import first.")


## Create small foreground tables and run basic link checks

This demonstrates how to identify missing references before importing larger files.


In [ ]:
assets = Path('tutorials/DAY 1/assets') if Path('tutorials/DAY 1/assets').exists() else Path('assets')
assets.mkdir(exist_ok=True)

activities = pd.DataFrame([
    {'code': 'FG1', 'name': 'Hydrogen use phase', 'unit': 'kg H2'},
    {'code': 'FG2', 'name': 'Electricity input', 'unit': 'kWh'},
])

exchanges = pd.DataFrame([
    {'from_code': 'FG1', 'to_code': 'FG2', 'type': 'technosphere', 'amount': 48.0},
    {'from_code': 'FG1', 'to_code': 'BIO_CO2', 'type': 'biosphere', 'amount': 0.2},
])

activities.to_csv(assets / 'foreground_activities.csv', index=False)
exchanges.to_csv(assets / 'foreground_exchanges.csv', index=False)

print('Wrote files to:', assets.resolve())


In [ ]:
activities = pd.read_csv(assets / 'foreground_activities.csv')
exchanges = pd.read_csv(assets / 'foreground_exchanges.csv')

known_codes = set(activities['code'])
unknown_tech_targets = [
    c for c, t in zip(exchanges['to_code'], exchanges['type'])
    if t == 'technosphere' and c not in known_codes
]

print('Known activity codes:', known_codes)
print('Unknown technosphere targets:', unknown_tech_targets)


## Exercise

Add one new exchange from `FG1` to an unknown code `FG3`, rerun the check,
and then fix the issue by adding activity `FG3`.


In [ ]:
# TODO


In [ ]:
exercises = exchanges.copy()
exercises.loc[len(exercises)] = {
    'from_code': 'FG1', 'to_code': 'FG3', 'type': 'technosphere', 'amount': 3.0
}

known_codes = set(activities['code'])
unknown_before = [
    c for c, t in zip(exercises['to_code'], exercises['type'])
    if t == 'technosphere' and c not in known_codes
]
print('Unknown targets before fix:', unknown_before)

activities_fixed = pd.concat([
    activities,
    pd.DataFrame([{'code': 'FG3', 'name': 'Water input', 'unit': 'kg'}])
], ignore_index=True)

known_codes_fixed = set(activities_fixed['code'])
unknown_after = [
    c for c, t in zip(exercises['to_code'], exercises['type'])
    if t == 'technosphere' and c not in known_codes_fixed
]
print('Unknown targets after fix:', unknown_after)
